## Assignment 3: Solving the Game of 24 Using Tree of Thoughts

- Student ID: 23520199
- Name: Cuong Chi Nguyen


# 1. Introduction

The **Game of 24** is a mathematical puzzle where the objective is to manipulate four integers using basic arithmetic operations ($+$, $-$, $\times$, $\div$) to reach the result **24**.

In this notebook, we explore two approaches to solve the Game of 24:

1. A classical Depth-First Search (DFS) algorithm that exhaustively explores all combinations of numbers and operations.
2. A Tree of Thoughts (ToT) prompting method using Large Language Models (LLMs) to heuristically guide the search process.

## References

[1] Shunyu Yao, Dian Yu, Jeffrey Zhao, Izhak Shafran, Thomas L. Griffiths, Yuan Cao, and Karthik Narasimhan. 2023. Tree of Thoughts: Deliberate Problem Solving with Large Language Models. Retrieved from https://arxiv.org/abs/2305.10601


# 2. Preparation


In [1]:
import operator
import time
import os
import asyncio
import nest_asyncio
import itertools
import logging
import re
import sympy as sp
from dotenv import load_dotenv
from typing import Iterable
from google import genai
from google.genai import types

In [ ]:
os.environ["GOOGLE_API_KEY"] = ""

In [3]:
load_dotenv()  # Load environment variables from .env file
nest_asyncio.apply()

os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
    handlers=[
        logging.FileHandler("logs/experiments.log", mode="a"),
    ],
)

In [4]:
# 10 tuples of test cases taken from https://www.4nums.com/game/difficulties/
TEST_CASES = [
    [11, 11, 12, 12],  # 24
    [1, 2, 6, 8],  # 193
    [1, 2, 10, 11],  # 237
    [5, 5, 6, 6],  # 414
    [9, 11, 13, 13],  # 548
    [6, 8, 8, 11],  # 731
    [2, 2, 7, 12],  # 934
    [6, 9, 9, 12],  # 1049
    [8, 9, 11, 11],  # 1257
    [2, 3, 5, 12],  # 1362
]

In [5]:
print(
    "Number of unique test cases:", len(set(tuple(sorted(case)) for case in TEST_CASES))
)

Number of unique test cases: 10


In [6]:
MAX_REQUESTS_PER_MINUTE = 25

LLM_CONFIG_B = {
    "model": "gemma-3-27b-it",
    "temperature": 1.0,
}

LLM_CONFIG_S = {
    "model": "gemma-3-12b-it",
    "temperature": 1.0,
}

# 3. Classical Search Methods


In [7]:
ops = {
    "+": operator.add,
    "-": operator.sub,
    "*": operator.mul,
    "/": operator.truediv,
    "-r": lambda x, y: operator.sub(y, x),  # Reverse subtraction
    "/r": lambda x, y: operator.truediv(y, x),  # Reverse division
}

In [8]:
def solve_go24_dfs(nums: Iterable[int], exps: Iterable[str]) -> tuple[bool, str]:

    if len(nums) == 1:
        return abs(nums[0] - 24) < 1e-6, exps[0]

    for i in range(len(nums) - 1):
        for j in range(i + 1, len(nums)):
            a, b = nums[i], nums[j]
            exp_a, exp_b = exps[i], exps[j]
            remaining_nums = [nums[k] for k in range(len(nums)) if k != i and k != j]
            remaining_exps = [exps[k] for k in range(len(exps)) if k != i and k != j]

            # Try all operations
            for op_name, op in ops.items():
                if (op_name == "/" and b == 0) or (op_name == "/r" and a == 0):
                    continue

                if op_name in ["-r", "/r"]:
                    new_exp = f"({exp_b} {op_name.strip('r')} {exp_a})"
                else:
                    new_exp = f"({exp_a} {op_name} {exp_b})"

                found, exp = solve_go24_dfs(
                    remaining_nums + [op(a, b)], remaining_exps + [new_exp]
                )

                if found:
                    return True, exp

    return False, ""

In [9]:
for i, nums in enumerate(TEST_CASES, start=1):
    exps = [str(num) for num in nums]
    found, exp = solve_go24_dfs(nums, exps)
    print(f"TC {i} - ", end="")
    if found:
        print(f"Found solution for {nums}: {exp[1:-1]} = 24")
    else:
        print(f"No solution found for {nums}")

TC 1 - Found solution for [11, 11, 12, 12]: (11 - 11) + (12 + 12) = 24
TC 2 - Found solution for [1, 2, 6, 8]: 8 * (6 - (1 + 2)) = 24
TC 3 - Found solution for [1, 2, 10, 11]: (1 + 2) + (10 + 11) = 24
TC 4 - Found solution for [5, 5, 6, 6]: 6 * ((5 + 5) - 6) = 24
TC 5 - Found solution for [9, 11, 13, 13]: (9 - 11) + (13 + 13) = 24
TC 6 - Found solution for [6, 8, 8, 11]: 8 * ((6 + 8) - 11) = 24
TC 7 - Found solution for [2, 2, 7, 12]: (2 * 7) - (2 - 12) = 24
TC 8 - Found solution for [6, 9, 9, 12]: (9 + 12) - (6 - 9) = 24
TC 9 - Found solution for [8, 9, 11, 11]: (11 * (11 - 8)) - 9 = 24
TC 10 - Found solution for [2, 3, 5, 12]: 12 / (3 - (5 / 2)) = 24


# 4. Tree of Thoughts (ToT) Prompting with LLMs

References: [princeton-nlp/tree-of-thought-llm](https://github.com/princeton-nlp/tree-of-thought-llm)


## 4.1. Prompts


In [10]:
standard_prompt = """Use numbers and basic arithmetic operations (+ - * /) to obtain 24. Each number used exactly once. Give the result in the format "Answer: ..." without any further explanation.
Input: 4 4 6 8
Answer: (4 + 8) * (6 - 4) = 24
Input: 2 9 10 12
Answer: 2 * 12 * (10 - 9) = 24
Input: 4 9 10 13
Answer: (13 - 9) * (10 - 4) = 24
Input: 1 4 8 8
Answer: (8 / 4 + 1) * 8 = 24
Input: 5 5 5 9
Answer: 5 + 5 + 5 + 9 = 24
Input: {input}
"""

In [11]:
cot_prompt = """Use numbers and basic arithmetic operations (+ - * /) to obtain 24. You have three steps: step 0 - input (4 nums), step 1 (3 nums), step 2 (2 nums), step 3 - output (1 num). Each step, you are only allowed to choose two of the remaining numbers from its previous step to obtain a new number. If the new number obtained by / operator is non-integer, keep it as a reduced fraction. Strictly follow the format demonstrated in the examples below.
Input: 4 4 6 8
Steps:
4 + 8 = 12 (left: 4 6 12)
6 - 4 = 2 (left: 2 12)
2 * 12 = 24 (left: 24)
Answer: (6 - 4) * (4 + 8) = 24
Input: 2 9 10 12
Steps:
12 * 2 = 24 (left: 9 10 24)
10 - 9 = 1 (left: 1 24)
24 * 1 = 24 (left: 24)
Answer: (12 * 2) * (10 - 9) = 24
Input: 4 9 10 13
Steps:
13 - 10 = 3 (left: 3 4 9)
9 - 3 = 6 (left: 4 6)
4 * 6 = 24 (left: 24)
Answer: 4 * (9 - (13 - 10)) = 24
Input: 1 4 8 8
Steps:
8 / 4 = 2 (left: 1 2 8)
1 + 2 = 3 (left: 3 8)
3 * 8 = 24 (left: 24)
Answer: (1 + 8 / 4) * 8 = 24
Input: {input}
Steps:
...
Answer: ...
"""

In [12]:
propose_prompt = """Strictly follow the format below to generate the next step for the given input. Each step, you are only allowed to choose two of the remaining numbers to obtain a new number. If the new number obtained by / operator is non-integer, keep it as a reduced fraction.
Input: 2 8 8 14
Possible next steps:
2 + 8 = 10 (left: 8 10 14)
8 / 2 = 4 (left: 4 8 14)
14 + 2 = 16 (left: 8 8 16)
2 * 8 = 16 (left: 8 14 16)
8 - 2 = 6 (left: 6 8 14)
14 - 8 = 6 (left: 2 6 8)
14 /  2 = 7 (left: 7 8 8)
14 / 8 = 7/4 (left: 7/4 2 8)
Input: {input}
Possible next steps:
"""

In [13]:
value_prompt = """Evaluate if given numbers can reach 24 with three following levels: sure, likely, impossible.

Input: 10 14
10 + 14 = 24
sure

Input: 11 12
11 + 12 = 23
12 - 11 = 1
11 * 12 = 132
11 / 12 = 0.91
impossible

Input: 4 4 10
4 + 4 + 10 = 8 + 10 = 18
4 * 10 - 4 = 40 - 4 = 36
(10 - 4) * 4 = 6 * 4 = 24
sure

Input: 4 9 11
9 + 11 + 4 = 20 + 4 = 24
sure

Input: 5 7 8
5 + 7 + 8 = 12 + 8 = 20
(8 - 5) * 7 = 3 * 7 = 21
I cannot obtain 24 now, but numbers are within a reasonable range
likely

Input: 5 6 6
5 + 6 + 6 = 17
(6 - 5) * 6 = 1 * 6 = 6
I cannot obtain 24 now, but numbers are within a reasonable range
likely

Input: 10 10 11
10 + 10 + 11 = 31
(11 - 10) * 10 = 10
10 10 10 are all too big
impossible

Input: 1 3 3
1 * 3 * 3 = 9
(1 + 3) * 3 = 12
1 3 3 are all too small
impossible

Input: {input}
"""

In [14]:
value_last_step_prompt = """Use numbers and basic arithmetic operations (+ - * /) to obtain 24. Given an input and an answer, give a judgement (sure/impossible) if the answer is correct, i.e. it uses each input exactly once and no other numbers, and reach 24.
Input: 4 4 6 8
Answer: (4 + 8) * (6 - 4) = 24
Judge: 
sure
Input: 2 9 10 12
Answer: 2 * 12 * (10 - 9) = 24
Judge: 
sure
Input: 4 9 10 13
Answer: (13 - 9) * (10 - 4) = 24
Judge: 
sure
Input: 4 4 6 8
Answer: (4 + 8) * (6 - 4) + 1 = 25
Judge: 
impossible
Input: 2 9 10 12
Answer: 2 * (12 - 10) = 24
Judge: 
impossible
Input: 4 9 10 13
Answer: (13 - 4) * (10 - 9) = 24
Judge: 
impossible
Input: {input}
Answer: {answer}
Judge:"""

## 4.2. LLM Wrapper


In [15]:
client = genai.Client().aio

RATE_LIMIT_SAFETY_MARGIN_SEC = 0.25

VALUE_CACHE: dict[tuple[str, str], float] = {}
PROPOSE_CACHE: dict[tuple[str, str], list[str]] = {}

In [16]:
_rate_limit_semaphore = asyncio.Semaphore(MAX_REQUESTS_PER_MINUTE)

TOTAL_API_CALLS = 0
TOTAL_TOKENS = 0


async def generate_text(prompt: str, model_config: dict, n: int = 1) -> list[str]:

    async def _pace_once() -> None:
        await _rate_limit_semaphore.acquire()

        async def _release_later():
            await asyncio.sleep(60.0 + RATE_LIMIT_SAFETY_MARGIN_SEC)
            _rate_limit_semaphore.release()

        asyncio.create_task(_release_later())

    async def _one_call() -> str:
        await _pace_once()
        response = await client.models.generate_content(
            model=model_config["model"],
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=model_config.get("temperature", 0.7),
                max_output_tokens=model_config.get("max_output_tokens", 512),
            ),
        )

        global TOTAL_API_CALLS, TOTAL_TOKENS
        TOTAL_API_CALLS += 1
        if hasattr(response, "usage_metadata") and response.usage_metadata:
            TOTAL_TOKENS += getattr(response.usage_metadata, "total_token_count", 0)

        return (response.text or "").strip()

    out: list[str] = []
    max_retries = 6

    for _ in range(n):
        last_err: Exception | None = None
        for attempt in range(max_retries + 1):
            try:
                out.append(await _one_call())
                break
            except Exception as e:
                last_err = e
                msg = str(e).lower()
                is_rate = ("429" in msg) or ("rate" in msg and "limit" in msg)

                if attempt == max_retries:
                    logging.error(f"LLM call failed after retries: {e}")
                    out.append("")
                    break

                backoff = min(8.0, (2**attempt) * 0.5)
                if is_rate:
                    backoff = max(backoff, 2.0)
                await asyncio.sleep(backoff)

        if last_err is not None and out[-1] == "":
            logging.warning(f"Returning empty text due to error: {last_err}")

    return out

## 4.3. Core logic


### 4.3.1. Utils


In [17]:
def get_current_numbers(y: str) -> str:
    last_line = y.strip().split("\n")[-1]
    if "left: " in last_line:
        return last_line.split("left: ")[-1].split(")")[0]
    return last_line


def _extract_value_label(text: str) -> str | None:
    if not text:
        return None
    t = text.strip().lower()
    last = t.splitlines()[-1].strip()

    for k in ("sure", "likely", "impossible"):
        if last == k:
            return k

    for k in ("sure", "likely", "impossible"):
        if k in t:
            return k
    return None


def _filter_step_lines(lines: list[str]) -> list[str]:
    """Keep only valid step lines that contain a '(left: ...)' state."""
    out: list[str] = []
    for line in lines:
        s = line.strip()
        if not s:
            continue
        low = s.lower()
        if low.startswith("possible next steps"):
            continue
        if low.startswith("input:"):
            continue
        if "left:" in low:
            out.append(s)
    return out


def _filter_answer_lines(lines: list[str]) -> list[str]:
    """Keep only plausible final answer lines."""
    out: list[str] = []
    for line in lines:
        s = line.strip()
        if not s:
            continue
        low = s.lower()
        if low.startswith("answer:") or ("=" in s):
            out.append(s)
    return out

### 4.3.2. ToT prompting with Beam Search


In [18]:
async def get_proposals(x: str, y: str, model_config: dict) -> list[str]:
    current_numbers = get_current_numbers(y if y else x)

    # If we already have left: 24, ask for the final Answer line.
    if current_numbers == "24":
        prompt = cot_prompt.format(input=x) + "Steps:" + y
        responses = await generate_text(prompt, model_config, n=1)
        text = responses[0] if responses else ""
        if not text:
            return []
        raw_lines = [ln for ln in text.split("\n") if ln.strip()]
        answer_lines = _filter_answer_lines(raw_lines)
        if not answer_lines:
            return []
        return [y + line + "\n" for line in answer_lines]

    # Otherwise: propose next steps from the current multiset
    prompt = propose_prompt.format(input=current_numbers)
    cache_key = (model_config["model"], prompt)
    if cache_key in PROPOSE_CACHE:
        lines = PROPOSE_CACHE[cache_key]
    else:
        responses = await generate_text(prompt, model_config, n=1)
        text = responses[0] if responses else ""
        if not text:
            return []
        raw_lines = [ln for ln in text.split("\n") if ln.strip()]
        step_lines = _filter_step_lines(raw_lines)
        if not step_lines:
            return []
        PROPOSE_CACHE[cache_key] = step_lines
        lines = step_lines

    return [y + line + "\n" for line in lines]


async def get_values(
    x: str, ys: list[str], model_config: dict, n_eval: int = 3
) -> list[float]:

    unique_ys: list[str] = []
    for y in ys:
        if y not in unique_ys:
            unique_ys.append(y)

    evaluated_values: dict[str, float] = {}
    prompt_to_ys: dict[str, list[str]] = {}

    for y in unique_ys:
        # if we already have 4 lines but no Answer, force value=0.
        if len(y.strip().split("\n")) == 4 and "answer" not in y.lower():
            evaluated_values[y] = 0.0
            continue

        last_line = y.strip().split("\n")[-1] if y.strip() else ""
        if "left: " not in last_line:  # last step
            ans = last_line.lower().replace("answer: ", "")
            prompt = value_last_step_prompt.format(input=x, answer=ans)
        else:
            current_numbers = get_current_numbers(y)
            prompt = value_prompt.format(input=current_numbers)

        cache_key = (model_config["model"], prompt)
        if cache_key in VALUE_CACHE:
            evaluated_values[y] = VALUE_CACHE[cache_key]
        else:
            prompt_to_ys.setdefault(prompt, []).append(y)

    prompts = list(prompt_to_ys.keys())
    if prompts:
        tasks = [generate_text(p, model_config, n=n_eval) for p in prompts]
        outputs_list = await asyncio.gather(*tasks)

        value_map = {"impossible": 0.001, "likely": 1.0, "sure": 20.0}

        for prompt, outs in zip(prompts, outputs_list):
            labels = [_extract_value_label(o) for o in (outs or [])]
            val = 0.0
            for lb in labels:
                if lb is None:
                    continue
                val += value_map.get(lb, 0.0)

            if val == 0.0:
                val = 0.001

            cache_key = (model_config["model"], prompt)
            VALUE_CACHE[cache_key] = val
            for y in prompt_to_ys[prompt]:
                evaluated_values[y] = val

    values: list[float] = []
    local_value_cache: dict[str, float] = {}
    for y in ys:
        if y in local_value_cache:
            values.append(0.0)
        else:
            v = evaluated_values.get(y, 0.0)
            local_value_cache[y] = v
            values.append(v)

    return values


async def solve_go24_tot_prompting(
    x: str, model_config: dict, max_steps=4, b=5, n_evaluate_sample=3
) -> list[str]:

    start = time.monotonic()

    ys = [""]

    for step in range(max_steps):
        proposal_tasks = [get_proposals(x, y, model_config) for y in ys]
        proposals_lists = await asyncio.gather(*proposal_tasks)

        new_ys = list(itertools.chain(*proposals_lists))
        if not new_ys:
            break

        new_ys = list(dict.fromkeys(new_ys))
        ids = list(range(len(new_ys)))

        values = await get_values(x, new_ys, model_config, n_eval=n_evaluate_sample)

        select_ids = sorted(ids, key=lambda i: values[i], reverse=True)[:b]
        ys = [new_ys[i] for i in select_ids]

        print(f"  [ToT Step {step+1}/{max_steps}] ")
        print(f"    total candidates: {len(new_ys)}")
        print(f"    selected: {ys}")
        print(f"    selected values: {[values[i] for i in select_ids]}")

        logging.info(
            f"[ToT Step {step+1}/{max_steps}] candidates={new_ys} selected={ys}"
        )

    end = time.monotonic()
    logging.info(f"Total time taken for {x}: {end - start}s")

    return ys

### 4.3.3. IO (Naive) Prompting


In [19]:
async def solve_go24_naive_prompting(x: str, model_config: dict) -> str:
    start = time.monotonic()
    prompt = standard_prompt.format(input=x)
    responses = await generate_text(prompt, model_config, n=1)
    end = time.monotonic()
    logging.info(f"Time taken for {x}: {end - start}s")
    return responses[0]

## 4.4. Evaluation Utils


In [ ]:
def test_output(x: str, output: str) -> int:
    if not output:
        return 0

    nums = list(map(int, x.split()))
    lines = [ln.strip() for ln in output.splitlines() if ln.strip()]
    if not lines:
        return 0

    # Prefer an explicit Answer line
    expr_line: str | None = None
    for ln in lines:
        if ln.lower().startswith("answer:"):
            expr_line = ln.split(":", 1)[1].strip()
            break

    # Fallback: any line containing "Answer:" somewhere
    if expr_line is None:
        for ln in lines:
            m = re.search(r"answer:\s*(.*)", ln, flags=re.IGNORECASE)
            if m:
                expr_line = m.group(1).strip()
                break

    # Fallback: last line with an equality
    if expr_line is None:
        for ln in reversed(lines):
            if "=" in ln:
                expr_line = ln
                break

    # Final fallback
    if expr_line is None:
        expr_line = lines[-1]

    # Strip any trailing "= ..." (e.g., "... = 24")
    expr_str = (
        expr_line.split("=", 1)[0].strip() if "=" in expr_line else expr_line.strip()
    )

    # Disallow decimal literals like "0.5" (not part of the original inputs)
    if re.search(r"\d+\.\d+", expr_str):
        return 0

    # Check for 4 numbers used, each exactly once
    used = list(map(int, re.findall(r"\d+", expr_str)))
    if sorted(used) != sorted(nums):
        return 0

    try:
        expr = sp.sympify(expr_str)
        val = sp.N(expr)
        return 1 if abs(float(val) - 24.0) < 1e-6 else 0
    except Exception:
        return 0

In [ ]:
# Quick check
print(test_output("1 2 6 8", "Answer: (2 * 1) * (6 / 2) * 8 = 24"))
print(test_output("1 2 6 8", "Answer: (2 * 1) * (6 / 2) = 24"))
print(test_output("1 2 6 8", "Answer: (2 * 1) * (6 / 8) = 24"))
print(test_output("1 2 6 8", "Answer: 6 /(2 * 1) * 8 = 24"))

0
0
0
1


## 4.5. Benchmarking


In [ ]:
async def run_experiment_io(test_cases: list[list[int]], model_config: dict):
    print(f"===========================================================")
    print(f"Evaluating {model_config.get('model', 'Unknown Model')} [IO]...")
    logging.info(
        f"Starting IO Prompting for {model_config.get('model', 'Unknown Model')}"
    )

    global TOTAL_API_CALLS, TOTAL_TOKENS
    TOTAL_API_CALLS = 0
    TOTAL_TOKENS = 0
    io_correct = 0

    start_time = time.monotonic()

    io_tasks = [
        solve_go24_naive_prompting(" ".join(map(str, nums)), model_config)
        for nums in TEST_CASES
    ]

    io_results = await asyncio.gather(*io_tasks)

    for i, (nums, ans) in enumerate(zip(TEST_CASES, io_results), start=1):
        x = " ".join(map(str, nums))
        is_io_correct = test_output(x, ans)
        io_correct += is_io_correct
        preview = ans.split("\n")[0][:50] if ans else "No Output"

        print(
            f"[IO Problem {i}] {x} -> Correct: {is_io_correct} | Output: {preview}..."
        )

        logging.info(
            f"[IO Problem {i}] {x} -> Correct: {is_io_correct} | Output: {preview}..."
        )

    end_time = time.monotonic()

    return {
        "model": model_config.get("model", "Unknown Model"),
        "io_correct": io_correct,
        "total_cases": len(TEST_CASES),
        "total_time_sec": end_time - start_time,
        "total_api_calls": TOTAL_API_CALLS,
        "total_tokens": TOTAL_TOKENS,
    }

In [ ]:
async def run_experiment_tot(test_cases: list[list[int]], model_config: dict):
    print(f"===========================================================")
    print(f"Evaluating {model_config.get('model', 'Unknown Model')} [ToT]...")
    logging.info(
        f"Starting ToT Prompting for {model_config.get('model', 'Unknown Model')}"
    )

    global TOTAL_API_CALLS, TOTAL_TOKENS
    TOTAL_API_CALLS = 0
    TOTAL_TOKENS = 0

    # Metrics to track:
    tot_correct_any = 0  # at least 1 out of b candidates is correct
    tot_avg = 0.0  # average correctness across all b candidates

    start_time = time.monotonic()

    for i, nums in enumerate(test_cases, start=1):
        x = " ".join(map(str, nums))

        print(f"[ToT Problem {i}] {x}")
        logging.info(f"[ToT Problem {i}] {x}")

        tot_ys = await solve_go24_tot_prompting(
            x, model_config, b=5, n_evaluate_sample=1
        )
        accs = [test_output(x, y) for y in (tot_ys or [])]

        any_correct = int(any(accs)) if accs else 0
        avg_correct = (sum(accs) / len(accs)) if accs else 0.0

        tot_correct_any += any_correct
        tot_avg += avg_correct

        print(
            f"-> [ToT Problem {i}] any={any_correct} avg={avg_correct:.3f} num_candidates={len(tot_ys)}"
        )
        logging.info(
            f"-> [ToT Problem {i}] any={any_correct} avg={avg_correct:.3f} num_candidates={len(tot_ys)}"
        )

    end_time = time.monotonic()

    return {
        "model": model_config.get("model", "Unknown Model"),
        "tot_correct_any": tot_correct_any,
        "tot_avg": tot_avg / len(test_cases),
        "total_time_sec": end_time - start_time,
        "total_api_calls": TOTAL_API_CALLS,
        "total_tokens": TOTAL_TOKENS,
    }

### 4.5.1. Benchmarking Bigger LLM


#### 4.5.1.1. IO Prompting


In [ ]:
b_result_io = asyncio.run(run_experiment_io(TEST_CASES, LLM_CONFIG_B))

Evaluating gemma-3-27b-it [IO]...
[IO Problem 1] 11 11 12 12 -> Correct: 0 | Output: Answer: 12 / (11 - 11) + 12 = 24...
[IO Problem 2] 1 2 6 8 -> Correct: 0 | Output: Answer: (8 - 6) * (2 + 1) = 24...
[IO Problem 3] 1 2 10 11 -> Correct: 0 | Output: Answer: (11 - 1) * (10 / 2) = 24...
[IO Problem 4] 5 5 6 6 -> Correct: 0 | Output: Answer: (6 / 5 + 1) * 6 = 24...
[IO Problem 5] 9 11 13 13 -> Correct: 0 | Output: Answer: 13 + 11 + 13 - 9 = 24...
[IO Problem 6] 6 8 8 11 -> Correct: 0 | Output: Answer: 8 * (11 - 8) - 6 = 24...
[IO Problem 7] 2 2 7 12 -> Correct: 0 | Output: Answer: (12 / 2) * (7 - 2) = 24...
[IO Problem 8] 6 9 9 12 -> Correct: 0 | Output: Answer: (12 / 6) * 9 + 9 = 24...
[IO Problem 9] 8 9 11 11 -> Correct: 0 | Output: Answer: 11 + 11 + 9 - 8 = 23...
[IO Problem 10] 2 3 5 12 -> Correct: 0 | Output: Answer: (12 / 2) * (5 - 3) = 24...


In [ ]:
print(f"===========================================================")
print(
    f"IO Prompting Accuracy of {b_result_io.get('model')}: {b_result_io.get('io_correct')}/{b_result_io.get('total_cases')}"
)
print(f"Total Time Taken: {b_result_io.get('total_time_sec'):.2f} seconds")
print(f"Total API Calls: {b_result_io.get('total_api_calls')}")
print(f"Total Tokens Used: {b_result_io.get('total_tokens')}")
print(f"===========================================================")

IO Prompting Accuracy of gemma-3-27b-it: 0/10
Total Time Taken: 10.89 seconds
Total API Calls: 10
Total Tokens Used: 2115


#### 4.5.1.2. ToT Prompting


In [25]:
b_result_tot = asyncio.run(run_experiment_tot(TEST_CASES, LLM_CONFIG_B))

Evaluating gemma-3-27b-it [ToT]...
[ToT Problem 1] 11 11 12 12
  [ToT Step 1/4] 
    total candidates: 9
    selected: ['11 + 12 = 23 (left: 11 12 23)\n', '11 - 11 = 0 (left: 0 12 12)\n', '12 - 11 = 1 (left: 1 11 12)\n', '11 / 11 = 1 (left: 1 12 12)\n', '12 / 11 = 12/11 (left: 12/11 11 12)\n']
    selected values: [20.0, 20.0, 20.0, 1.0, 1.0]
  [ToT Step 2/4] 
    total candidates: 48
    selected: ['11 + 12 = 23 (left: 11 12 23)\n12 + 23 = 35 (left: 11 35)\n', '11 + 12 = 23 (left: 11 12 23)\n12 - 11 = 1 (left: 1 23)\n', '11 + 12 = 23 (left: 11 12 23)\n23 - 11 = 12 (left: 12 12)\n', '11 - 11 = 0 (left: 0 12 12)\n0 + 12 = 12 (left: 12 12)\n', '11 - 11 = 0 (left: 0 12 12)\n12 / 0 = undefined (left: 12 12)\n']
    selected values: [20.0, 20.0, 20.0, 20.0, 20.0]
  [ToT Step 3/4] 
    total candidates: 20
    selected: ['11 + 12 = 23 (left: 11 12 23)\n12 + 23 = 35 (left: 11 35)\n11 + 35 = 46 (left: 46)\n', '11 + 12 = 23 (left: 11 12 23)\n12 + 23 = 35 (left: 11 35)\n35 / 11 = 35/11 (left: 35

In [26]:
print(f"===========================================================")
print(f"ToT Prompting Accuracy of {b_result_tot.get('model')}:")
print(f"- any = {b_result_tot.get('tot_correct_any')}/{len(TEST_CASES)}")
print(f"- avg = {b_result_tot.get('tot_avg'):.3f}")
print(f"Total Time Taken: {b_result_tot.get('total_time_sec'):.2f} seconds")
print(f"Total API Calls: {b_result_tot.get('total_api_calls')}")
print(f"Total Tokens Used: {b_result_tot.get('total_tokens')}")
print(f"===========================================================")

ToT Prompting Accuracy of gemma-3-27b-it:
- any = 7/10
- avg = 0.382
Total Time Taken: 1778.89 seconds
Total API Calls: 729
Total Tokens Used: 300703


### 4.5.2. Benchmarking Smaller LLM


#### 4.5.2.1. IO Prompting


In [ ]:
s_result_io = asyncio.run(run_experiment_io(TEST_CASES, LLM_CONFIG_S))

Evaluating gemma-3-12b-it [IO]...
[IO Problem 1] 11 11 12 12 -> Correct: 0 | Output: Answer: 12 * (12 / 11 + 11 / 11) = 24...
[IO Problem 2] 1 2 6 8 -> Correct: 0 | Output: Answer: 8 * (6 - 2) - 1 = 23...
[IO Problem 3] 1 2 10 11 -> Correct: 0 | Output: Answer: (11 - 1) * (10 / 2) = 24...
[IO Problem 4] 5 5 6 6 -> Correct: 0 | Output: Answer: (6 - 5) * (6 * 5) = 30...
[IO Problem 5] 9 11 13 13 -> Correct: 0 | Output: Answer: (13 - 11) * (13 + 9) = 24...
[IO Problem 6] 6 8 8 11 -> Correct: 0 | Output: Answer: (11 - 8) * (8 - 6) = 12...
[IO Problem 7] 2 2 7 12 -> Correct: 0 | Output: Answer: (12 / 2) * (7 - 2) = 24...
[IO Problem 8] 6 9 9 12 -> Correct: 0 | Output: Answer: (12 / 6) * 9 + 9 = 24...
[IO Problem 9] 8 9 11 11 -> Correct: 0 | Output: Answer: (11 - 9) * (8 + 11) = 24...
[IO Problem 10] 2 3 5 12 -> Correct: 0 | Output: Answer: (12 / 3) * (5 + 2) = 24...


In [ ]:
print(f"===========================================================")
print(
    f"IO Prompting Accuracy of {s_result_io.get('model')}: {s_result_io.get('io_correct')}/{s_result_io.get('total_cases')}"
)
print(f"Total Time Taken: {s_result_io.get('total_time_sec'):.2f} seconds")
print(f"Total API Calls: {s_result_io.get('total_api_calls')}")
print(f"Total Tokens Used: {s_result_io.get('total_tokens')}")
print(f"===========================================================")

IO Prompting Accuracy of gemma-3-12b-it: 0/10
Total Time Taken: 5.33 seconds
Total API Calls: 10
Total Tokens Used: 2115


#### 4.5.2.2. ToT Prompting


In [29]:
s_result_tot = asyncio.run(run_experiment_tot(TEST_CASES, LLM_CONFIG_S))

Evaluating gemma-3-12b-it [ToT]...
[ToT Problem 1] 11 11 12 12
  [ToT Step 1/4] 
    total candidates: 9
    selected: ['11 * 11 = 121 (left: 11 12 121)\n', '11 * 12 = 132 (left: 11 12 132)\n', '11 - 11 = 0 (left: 0 12 12)\n', '12 - 11 = 1 (left: 11 12 1)\n', '11 / 11 = 1 (left: 1 12 12)\n']
    selected values: [20.0, 20.0, 20.0, 20.0, 1.0]
  [ToT Step 2/4] 
    total candidates: 28
    selected: ['11 * 11 = 121 (left: 11 12 121)\n11 + 12 = 23 (left: 12 121 23)\n', '11 * 11 = 121 (left: 11 12 121)\n11 * 12 = 132 (left: 12 121 132)\n', '11 * 11 = 121 (left: 11 12 121)\n12 + 121 = 133 (left: 11 12 133)\n', '11 * 11 = 121 (left: 11 12 121)\n12 - 11 = 1 (left: 1 12 121)\n', '11 * 11 = 121 (left: 11 12 121)\n121 - 12 = 109 (left: 11 12 109)\n']
    selected values: [20.0, 20.0, 20.0, 20.0, 20.0]
  [ToT Step 3/4] 
    total candidates: 55
    selected: ['11 * 11 = 121 (left: 11 12 121)\n11 + 12 = 23 (left: 12 121 23)\n121 + 23 = 144 (left: 12 144)\n', '11 * 11 = 121 (left: 11 12 121)\n11 + 

In [30]:
print(f"===========================================================")
print(f"ToT Prompting Accuracy of {s_result_tot.get('model')}:")
print(f"- any = {s_result_tot.get('tot_correct_any')}/{len(TEST_CASES)}")
print(f"- avg = {s_result_tot.get('tot_avg'):.3f}")
print(f"Total Time Taken: {s_result_tot.get('total_time_sec'):.2f} seconds")
print(f"Total API Calls: {s_result_tot.get('total_api_calls')}")
print(f"Total Tokens Used: {s_result_tot.get('total_tokens')}")
print(f"===========================================================")

ToT Prompting Accuracy of gemma-3-12b-it:
- any = 3/10
- avg = 0.100
Total Time Taken: 2130.10 seconds
Total API Calls: 837
Total Tokens Used: 343450


# 5. Additional Tests


In [ ]:
new_client = genai.Client()

In [ ]:
# I'll use the hardest test case for a quick evalution of IO prompting with SOTA LLMs.
hardest_case = TEST_CASES[-1]
x = " ".join(map(str, hardest_case))
prompt = standard_prompt.format(input=x)
prompt

'Use numbers and basic arithmetic operations (+ - * /) to obtain 24. Each number used exactly once. Give the result in the format "Answer: ..." without any further explanation.\nInput: 4 4 6 8\nAnswer: (4 + 8) * (6 - 4) = 24\nInput: 2 9 10 12\nAnswer: 2 * 12 * (10 - 9) = 24\nInput: 4 9 10 13\nAnswer: (13 - 9) * (10 - 4) = 24\nInput: 1 4 8 8\nAnswer: (8 / 4 + 1) * 8 = 24\nInput: 5 5 5 9\nAnswer: 5 + 5 + 5 + 9 = 24\nInput: 2 3 5 12\n'

In [ ]:
models = [
    "gemma-4-31b-it",
    "gemma-4-26b-a4b-it",
]
responses = []
costs = []

In [ ]:
for model in models:
    start = time.monotonic()
    response = new_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.7,
        ),
    )
    time_taken = time.monotonic() - start
    costs.append(
        {
            "time_taken_sec": time_taken,
            "total_tokens": (
                response.usage_metadata.total_token_count
                if response.usage_metadata
                else None
            ),
        }
    )
    responses.append(response)

In [ ]:
print(f"Model {models[0]} response:\n{responses[0]}\n")

Model gemma-4-31b-it response:
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""*   Numbers: 2, 3, 5, 12
    *   Target: 24
    *   Constraints: Use each number exactly once, basic arithmetic operations (+, -, *, /).

    *   24 = 12 * 2
    *   24 = 8 * 3
    *   24 = 6 * 4
    *   24 = 4 * 6
    *   24 = 12 + 12
    *   24 = 18 + 6
    *   24 = 20 + 4
    *   24 = 29 - 5
    *   24 = 30 - 6

    *   Using 12 * 2:
        *   (5 - 3) * 12 = 2 * 12 = 24 (Numbers: 5, 3, 12; 2 is left over) - *Wait, I must use all numbers.*
        *   Numbers: 2, 3, 5, 12.
        *   Try: 12 * (5 - 3) = 24. (Still missing the 2).
        *   Try: 12 * 2 = 24. (Using 3, 5 to get 1? 5-3=2, 3-2=1... no).
        *   Try: (5 - 3) * 12 = 24. Wait, the numbers are 2, 3, 5, 12.
        *   If I use (5 - 3) * 12, I've used 5, 3, 12. I still have 2. This doesn't work.

    *   Using 8 * 3:
        *   Can we get 8 from 2,

In [ ]:
print(f"Model {models[1]} response:\n{responses[1]}\n")

Model gemma-4-26b-a4b-it response:
sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""*   Input numbers: 2, 3, 5, 12
    *   Target number: 24
    *   Rules: Use each number exactly once, basic arithmetic operations (+, -, *, /).

    *   2 * 12 = 24 (Need to use 3 and 5 to get 1)
        *   5 - 3 = 2 (No)
        *   (5 - 3) / ? (No)
        *   Wait, 5 - 3 = 2, 2 * 12 = 24, but we still have the extra 2.
        *   Wait, 2 * 12 = 24. Can I use 5 and 3 to get 1?
        *   (5 - 3) = 2. No.
        *   (5 - 3) - 1? (No 1 available).
    *   3 * 8 = 24 (Need to use 2, 5, 12 to get 8)
        *   12 - 5 + 2 = 9 (No)
        *   12 - (5 - 2) = 9 (No)
        *   (12 / 2) + 5 = 11 (No)
        *   12 - 5 + 1 = 8? No 1.
        *   12 - (5 - 2)? No.
        *   12 - 2 - 5 = 5 (No)
        *   Maybe 12 / 2 + 5 - 3? No.
    *   4 * 6 = 24 (Need to use 2, 3, 5, 12 to get 4 and 6)
        *   12 / (5 - 2

In [ ]:
for i, (model, response) in enumerate(zip(models, responses)):
    ans = response.text.strip() if response.text else ""
    is_correct = test_output(x, ans)
    print(f"Model: {model} | Answer: {ans} | Correct: {is_correct}")
    print(f"  Time taken: {costs[i]['time_taken_sec']:.2f} seconds")
    print(f"  Total tokens: {costs[i]['total_tokens']}")

Model: gemma-4-31b-it | Answer: Answer: 12 / (3 - 5 / 2) = 24 | Correct: 1
  Time taken: 346.33 seconds
  Total tokens: 12437
Model: gemma-4-26b-a4b-it | Answer:  | Correct: 0
  Time taken: 689.48 seconds
  Total tokens: 32976
